## Configuration ##

In [5]:
import os
import json
from pathlib import Path
import pandas as pd
import requests
from dbrepo.RestClient import RestClient
from dbrepo.api.dto import AccessType, QueryDefinition, JoinDefinition, JoinType, ConditionalDefinition, CreateView, Subset, SubsetColumn, Join, Conditional

def find_repo_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'data' / 'raw').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root containing data/raw')

load_env = os.getenv('DBREPO_USER') is None or os.getenv('DBREPO_PASSWORD') is None
if load_env:
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except Exception:
        pass

import os
import json
import pathlib
import requests
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import numpy as np
from dbrepo.RestClient import RestClient
load_dotenv()

USERNAME = os.getenv("DBREPO_USER")
PASSWORD = os.getenv("DBREPO_PASSWORD")

auth = (USERNAME, PASSWORD)
HOST = "https://test.dbrepo.tuwien.ac.at"
PORT = 3306

API_BASE  = 'https://test.dbrepo.tuwien.ac.at/api/v1'
DB_NAME = "data_stewardship_group6_crash_serverity_prediction_lkhb"
DB_ID = "3d81c073-e5fd-49b9-9536-b75ed490ca3e"
CONTAINER_ID = '6cfb3b8e-1792-4e46-871a-f3d103527203'


print('Configuration loaded.')
ROOT = find_repo_root()
RAW_DIR = ROOT / 'data' / 'raw'
client = RestClient(endpoint=HOST, username=USERNAME, password=PASSWORD, secure=True)
print(f'Configuration loaded for {DB_NAME}.')

Configuration loaded.
Configuration loaded for data_stewardship_group6_crash_serverity_prediction_lkhb.


## Create database and tables ##

In [6]:
def get_or_create_database(client, name, container_id):
    for database_brief in client.get_databases():
        if database_brief.name == name:
            print(f'Using existing database: {database_brief.id}')
            return client.get_database(database_brief.id)

    try:
        database = client.create_database(name=name, container_id=container_id, is_public=True, is_schema_public=True)
        database_id = getattr(database, 'id', None) or database.get('id')
        print(f'Created database: {database_id}')
        return client.get_database(database_id)
    except Exception as exc:
        print(f'create_database via client failed, trying to resolve by name: {exc}')
        for database_brief in client.get_databases():
            if database_brief.name == name:
                print(f'Using database created during the failed call: {database_brief.id}')
                return client.get_database(database_brief.id)
        response = requests.post(
            f"{HOST}/api/v1/database", 
            auth=(USERNAME, PASSWORD),
            headers={"Content-Type": "application/json", "Accept": "application/json"},
            json={"name": name, "container_id": container_id, "is_public": True, "is_schema_public": True},
            verify=True,
        )
        if response.status_code not in (200, 201):
            raise RuntimeError(f'Failed to create database via HTTP: {response.status_code} {response.text}')
        created = response.json()
        database_id = created.get('id') or created.get('database_id')
        print(f'Created database via HTTP: {database_id}')
        return client.get_database(database_id)


def load_table_dataframe(csv_name, index_name, join_key=False):
    dataframe = pd.read_csv(RAW_DIR / csv_name)
    dataframe = dataframe.copy()
    dataframe[index_name] = range(1, len(dataframe) + 1)
    if join_key:
        dataframe['join_key'] = dataframe['collision_index'].astype(str) + '|' + dataframe['vehicle_reference'].astype(str)
    dataframe = dataframe.set_index(index_name)
    return dataframe


database = get_or_create_database(client, DB_NAME, CONTAINER_ID)
DATABASE_ID = database.id
print(f'Database ready: {database.name} ({DATABASE_ID})')

try:
    client.update_database_schema(DATABASE_ID)
    print('Database schema synchronized.')
except Exception as exc:
    print(f'update_database_schema returned: {exc}')

# Recreate the three base tables in the test database so the technical join_key exists on both
# casualty and vehicle. This keeps the notebook self-contained for view creation tests.
base_tables = ['collision', 'vehicle', 'casualty']
for table in list(client.get_tables(DATABASE_ID)):
    if table.name in base_tables:
        try:
            client.delete_table(DATABASE_ID, table.id)
            print(f'Deleted existing table: {table.name}')
        except Exception as exc:
            print(f'Could not delete table {table.name}: {exc}')

# Refresh table metadata after deletions.
table_specs = [
    ('collision', 'stats19-collision-2023-raw-v1.csv', 'One row per road collision reported to the police in Great Britain during 2023.', 'collision_index', False),
    ('vehicle', 'stats19-vehicle-2023-raw-v1.csv', 'One row per vehicle involved in a recorded collision.', 'vehicle_id', True),
    ('casualty', 'stats19-casualty-2023-raw-v1.csv', 'One row per casualty in a recorded collision.', 'casualty_id', True),
]

for table_name, csv_name, description, index_name, add_join_key in table_specs:
    dataframe = load_table_dataframe(csv_name, index_name, join_key=add_join_key)
    try:
        table = client.create_table(
            database_id=DATABASE_ID,
            name=table_name,
            is_public=True,
            is_schema_public=True,
            dataframe=dataframe,
            description=description,
            with_data=False,
        )
        print(f'Created table schema {table_name}: {table.id}')
        try:
            client.import_table_data(DATABASE_ID, table.id, dataframe.reset_index())
            print(f'Imported data for table {table_name}')
        except Exception as exc:
            print(f'Failed to import data for table {table_name}: {exc}')
    except Exception as exc:
        print(f'Failed to create table {table_name}: {exc}')

create_database via client failed, trying to resolve by name: 3 validation errors for Database
is_dashboard_enabled
  Field required [type=missing, input_value={'id': 'ea768898-e9ee-4db..., 'preview_image': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.8/v/missing
container
  Field required [type=missing, input_value={'id': 'ea768898-e9ee-4db..., 'preview_image': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.8/v/missing
owner
  Field required [type=missing, input_value={'id': 'ea768898-e9ee-4db..., 'preview_image': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.8/v/missing
Using database created during the failed call: ea768898-e9ee-4db7-83fe-6d533a85ef9e
Database ready: data_stewardship_group6_crash_serverity_prediction_lkhb (ea768898-e9ee-4db7-83fe-6d533a85ef9e)
update_database_schema returned: 1 validation error for DatabaseBrief
contact
  Input should be a va

C:\Users\Sebastian\AppData\Local\Temp\ipykernel_15548\3958237598.py:34: DtypeWarning: Columns (27) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframe = pd.read_csv(RAW_DIR / csv_name)


Created table schema vehicle: 19b16588-e8a9-41ab-9b18-225b1afc6d14
Imported data for table vehicle
Created table schema casualty: 21802163-fdee-4e4d-a71b-d2e9c2884fb7
Imported data for table casualty


## Create the ML view ##

In [7]:
import requests

# Refresh database metadata to ensure we have the latest tables
database = client.get_database(DATABASE_ID)
db = database

# Create v_severity_distribution (mono-table casualty - TESTED AND PASSING)
print("Creating v_severity_distribution (casualty mono-table)...")
cas_brief = next(t for t in db.tables if t.name == 'casualty')
cas = client.get_table(DATABASE_ID, cas_brief.id)
def cid(table_obj, name):
    return next(c.id for c in table_obj.columns if c.internal_name == name)

subset_sev = Subset(
    datasource_ids=[cas.id],
    columns=[SubsetColumn(id=cid(cas,'casualty_severity'), alias='casualty_severity')],
    joins=None,
    filters=None,
    orders=None,
)
payload_sev = CreateView(name='v_severity_distribution', query=subset_sev, is_public=True, is_schema_public=True).model_dump(mode='json', exclude_none=True)
r_sev = requests.post(f'{HOST}/api/v1/database/{DATABASE_ID}/view', auth=(USERNAME, PASSWORD), headers={'Content-Type':'application/json','Accept':'application/json'}, json=payload_sev)
print(f"  v_severity_distribution: {r_sev.status_code} {'✓' if r_sev.status_code in (200,201) else '✗'}")

# Create v_collision_summary (simplified: collision mono-table only, no complex aggregations)
print("Creating v_collision_summary (collision mono-table)...")
col_brief = next(t for t in db.tables if t.name == 'collision')
col = client.get_table(DATABASE_ID, col_brief.id)
subset_col = Subset(
    datasource_ids=[col.id],
    columns=[
        SubsetColumn(id=cid(col,'collision_index'), alias='collision_index'),
        SubsetColumn(id=cid(col,'date'), alias='date'),
        SubsetColumn(id=cid(col,'day_of_week'), alias='day_of_week'),
        SubsetColumn(id=cid(col,'time'), alias='time'),
        SubsetColumn(id=cid(col,'road_type'), alias='road_type'),
        SubsetColumn(id=cid(col,'speed_limit'), alias='speed_limit'),
        SubsetColumn(id=cid(col,'weather_conditions'), alias='weather_conditions'),
        SubsetColumn(id=cid(col,'light_conditions'), alias='light_conditions'),
        SubsetColumn(id=cid(col,'road_surface_conditions'), alias='road_surface_conditions'),
        SubsetColumn(id=cid(col,'number_of_vehicles'), alias='number_of_vehicles'),
        SubsetColumn(id=cid(col,'number_of_casualties'), alias='number_of_casualties'),
    ],
    joins=None,
    filters=None,
    orders=None,
)
payload_col = CreateView(name='v_collision_summary', query=subset_col, is_public=True, is_schema_public=True).model_dump(mode='json', exclude_none=True)
r_col = requests.post(f'{HOST}/api/v1/database/{DATABASE_ID}/view', auth=(USERNAME, PASSWORD), headers={'Content-Type':'application/json','Accept':'application/json'}, json=payload_col)
print(f"  v_collision_summary: {r_col.status_code} {'✓' if r_col.status_code in (200,201) else '✗'}")

# Create v_feature_null_check (simplified: casualty mono-table only)
print("Creating v_feature_null_check (casualty mono-table)...")
subset_null = Subset(
    datasource_ids=[cas.id],
    columns=[
        SubsetColumn(id=cid(cas,'casualty_id'), alias='casualty_id'),
        SubsetColumn(id=cid(cas,'casualty_severity'), alias='casualty_severity'),
        SubsetColumn(id=cid(cas,'age_of_casualty'), alias='age_of_casualty'),
        SubsetColumn(id=cid(cas,'casualty_type'), alias='casualty_type'),
    ],
    joins=None,
    filters=None,
    orders=None,
)
payload_null = CreateView(name='v_feature_null_check', query=subset_null, is_public=True, is_schema_public=True).model_dump(mode='json', exclude_none=True)
r_null = requests.post(f'{HOST}/api/v1/database/{DATABASE_ID}/view', auth=(USERNAME, PASSWORD), headers={'Content-Type':'application/json','Accept':'application/json'}, json=payload_null)
print(f"  v_feature_null_check: {r_null.status_code} {'✓' if r_null.status_code in (200,201) else '✗'}")

print("\nAll 3 views created successfully.")

Creating v_severity_distribution (casualty mono-table)...
  v_severity_distribution: 201 ✓
Creating v_collision_summary (collision mono-table)...
  v_collision_summary: 201 ✓
Creating v_feature_null_check (casualty mono-table)...
  v_feature_null_check: 201 ✓

All 3 views created successfully.


In [8]:
# Create materialized table t_ml_features (join of casualty + collision + vehicle)
print("\nCreating materialized table t_ml_features...")

# Reload with correct dtype
collision_df = pd.read_csv(RAW_DIR / 'stats19-collision-2023-raw-v1.csv')
vehicle_df = pd.read_csv(RAW_DIR / 'stats19-vehicle-2023-raw-v1.csv', dtype={'vehicle_reference': float})
casualty_df = pd.read_csv(RAW_DIR / 'stats19-casualty-2023-raw-v1.csv')

print(f"Loaded: collision ({len(collision_df):,}), vehicle ({len(vehicle_df):,}), casualty ({len(casualty_df):,})")

# Join in Python: casualty -> collision -> vehicle
# Casualty key is: (collision_index, vehicle_reference, casualty_reference)
ml_features = casualty_df[[
    'collision_index', 'vehicle_reference', 'casualty_reference', 'casualty_severity',
    'age_of_casualty', 'casualty_type'
]].copy()

ml_features = ml_features.merge(
    collision_df[[
        'collision_index', 'road_type', 'speed_limit', 'weather_conditions', 
        'light_conditions', 'road_surface_conditions', 'time', 'day_of_week', 'number_of_vehicles'
    ]],
    on='collision_index',
    how='inner'
)

ml_features = ml_features.merge(
    vehicle_df[['collision_index', 'vehicle_reference', 'vehicle_type']],
    on=['collision_index', 'vehicle_reference'],
    how='inner'
)

print(f"After joining: {len(ml_features):,} rows")
print(f"Columns: {list(ml_features.columns)}")

# Create schema for t_ml_features with a proper index
ml_features_schema = ml_features.copy()
ml_features_schema.insert(0, 'ml_feature_id', range(1, len(ml_features_schema) + 1))
ml_features_schema = ml_features_schema.set_index('ml_feature_id')

# Delete existing t_ml_features if present
existing = client.get_tables(DATABASE_ID)
for tbl in existing:
    if tbl.name == 't_ml_features':
        try:
            client.delete_table(DATABASE_ID, tbl.id)
            print(f"Deleted existing t_ml_features")
        except Exception as e:
            print(f"Could not delete t_ml_features: {e}")

# Create t_ml_features schema
try:
    ml_table = client.create_table(
        database_id=DATABASE_ID,
        name='t_ml_features',
        is_public=True,
        is_schema_public=True,
        dataframe=ml_features_schema,
        description='Materialized ML features table: casualty + collision + vehicle joined.',
        with_data=False,
    )
    print(f'Created t_ml_features schema: {ml_table.id}')
    
    # Import the data
    try:
        client.import_table_data(DATABASE_ID, ml_table.id, ml_features.reset_index(drop=True))
        print(f'Imported {len(ml_features):,} rows into t_ml_features')
    except Exception as exc:
        print(f'Failed to import t_ml_features data: {exc}')
except Exception as exc:
    print(f'Failed to create t_ml_features: {exc}')

print("\nMaterialized table creation complete.")


Creating materialized table t_ml_features...


C:\Users\Sebastian\AppData\Local\Temp\ipykernel_15548\982225542.py:6: DtypeWarning: Columns (27) have mixed types. Specify dtype option on import or set low_memory=False.
  vehicle_df = pd.read_csv(RAW_DIR / 'stats19-vehicle-2023-raw-v1.csv', dtype={'vehicle_reference': float})


Loaded: collision (104,258), vehicle (189,815), casualty (132,977)
After joining: 132,977 rows
Columns: ['collision_index', 'vehicle_reference', 'casualty_reference', 'casualty_severity', 'age_of_casualty', 'casualty_type', 'road_type', 'speed_limit', 'weather_conditions', 'light_conditions', 'road_surface_conditions', 'time', 'day_of_week', 'number_of_vehicles', 'vehicle_type']
Created t_ml_features schema: 7107fdfa-bcb8-4150-a1ac-244eae814efd
Failed to import t_ml_features data: Failed to insert table data: data service failed to establish connection to metadata service

Materialized table creation complete.


In [9]:
# Check actual column names in CSV files
import pandas as pd
from pathlib import Path

casualty_df = pd.read_csv(RAW_DIR / 'stats19-casualty-2023-raw-v1.csv')
collision_df = pd.read_csv(RAW_DIR / 'stats19-collision-2023-raw-v1.csv')
vehicle_df = pd.read_csv(RAW_DIR / 'stats19-vehicle-2023-raw-v1.csv')

print("Casualty columns:", casualty_df.columns.tolist())
print("\nCollision columns:", collision_df.columns.tolist())
print("\nVehicle columns:", vehicle_df.columns.tolist())
print("\nCasualty sample:")
print(casualty_df.head(2))
print("\nVehicle sample:")
print(vehicle_df.head(2))

Casualty columns: ['collision_index', 'collision_year', 'collision_ref_no', 'vehicle_reference', 'casualty_reference', 'casualty_class', 'sex_of_casualty', 'age_of_casualty', 'age_band_of_casualty', 'casualty_severity', 'pedestrian_location', 'pedestrian_movement', 'car_passenger', 'bus_or_coach_passenger', 'pedestrian_road_maintenance_worker', 'casualty_type', 'casualty_imd_decile', 'lsoa_of_casualty', 'enhanced_casualty_severity', 'casualty_injury_based', 'casualty_adjusted_severity_serious', 'casualty_adjusted_severity_slight', 'casualty_distance_banding']

Collision columns: ['collision_index', 'collision_year', 'collision_ref_no', 'location_easting_osgr', 'location_northing_osgr', 'longitude', 'latitude', 'police_force', 'collision_severity', 'number_of_vehicles', 'number_of_casualties', 'date', 'day_of_week', 'time', 'local_authority_district', 'local_authority_ons_district', 'local_authority_highway', 'local_authority_highway_current', 'first_road_class', 'first_road_number', 'r

C:\Users\Sebastian\AppData\Local\Temp\ipykernel_15548\3701632940.py:7: DtypeWarning: Columns (27) have mixed types. Specify dtype option on import or set low_memory=False.
  vehicle_df = pd.read_csv(RAW_DIR / 'stats19-vehicle-2023-raw-v1.csv')


In [10]:
# Check if casualty_id exists, and what columns are available
print("Casualty columns:")
print(casualty_df.columns.tolist()[:20])  # First 20 cols
print("\nVehicle columns:")  
print(vehicle_df.columns.tolist()[:20])  # First 20 cols

# Check for ID columns
print("\nLooking for ID columns in casualty:")
id_cols = [col for col in casualty_df.columns if 'id' in col.lower()]
print(id_cols)

print("\nLooking for ID columns in vehicle:")
id_cols = [col for col in vehicle_df.columns if 'id' in col.lower()]
print(id_cols)

# Check first row to understand structure
print("\nCasualty first row keys:")
print(casualty_df.iloc[0].to_dict())

Casualty columns:
['collision_index', 'collision_year', 'collision_ref_no', 'vehicle_reference', 'casualty_reference', 'casualty_class', 'sex_of_casualty', 'age_of_casualty', 'age_band_of_casualty', 'casualty_severity', 'pedestrian_location', 'pedestrian_movement', 'car_passenger', 'bus_or_coach_passenger', 'pedestrian_road_maintenance_worker', 'casualty_type', 'casualty_imd_decile', 'lsoa_of_casualty', 'enhanced_casualty_severity', 'casualty_injury_based']

Vehicle columns:
['collision_index', 'collision_year', 'collision_ref_no', 'vehicle_reference', 'vehicle_type', 'towing_and_articulation', 'vehicle_manoeuvre_historic', 'vehicle_manoeuvre', 'vehicle_direction_from', 'vehicle_direction_to', 'vehicle_location_restricted_lane_historic', 'vehicle_location_restricted_lane', 'junction_location', 'skidding_and_overturning', 'hit_object_in_carriageway', 'vehicle_leaving_carriageway', 'hit_object_off_carriageway', 'first_point_of_impact', 'vehicle_left_hand_drive', 'journey_purpose_of_drive

In [11]:
# Debug views issue - check what's in DBRepo now
print("Checking current database state...")
db = client.get_database(DATABASE_ID)
print(f"Database: {db.name}")
print(f"Tables ({len(db.tables)}):")
for t in db.tables:
    print(f"  - {t.name} (id: {t.id})")

print(f"\nViews ({len(db.views)}):")
for v in db.views:
    print(f"  - {v.name} (id: {v.id})")

# Now let's check one of the tables in detail
cas_table = next((t for t in db.tables if t.name == 'casualty'), None)
if cas_table:
    cas_detail = client.get_table(DATABASE_ID, cas_table.id)
    print(f"\nCasualty table detail:")
    print(f"  ID: {cas_detail.id}")
    print(f"  Columns ({len(cas_detail.columns)}):")
    for i, col in enumerate(cas_detail.columns[:5]):  # Show first 5
        print(f"    - {col.name} (id: {col.id}, internal: {col.internal_name})")

Checking current database state...
Database: data_stewardship_group6_crash_serverity_prediction_lkhb
Tables (4):
  - t_ml_features (id: 7107fdfa-bcb8-4150-a1ac-244eae814efd)
  - casualty (id: 21802163-fdee-4e4d-a71b-d2e9c2884fb7)
  - vehicle (id: 19b16588-e8a9-41ab-9b18-225b1afc6d14)
  - collision (id: f128f1d4-2de7-46e7-8057-f28e6d860d88)

Views (3):
  - v_feature_null_check (id: 5fc3c297-c8ab-4e57-b110-513ec24ffec1)
  - v_collision_summary (id: fedb5130-e112-4b66-ab74-607ad0984627)
  - v_severity_distribution (id: 31267453-1891-4227-870f-e2d72644811d)

Casualty table detail:
  ID: 21802163-fdee-4e4d-a71b-d2e9c2884fb7
  Columns (25):
    - casualty_id (id: 7d01c01a-3d0b-426b-9a16-490332b98a25, internal: casualty_id)
    - collision_index (id: 4587b385-f2dd-4688-9f44-90bbf3b88e4b, internal: collision_index)
    - collision_year (id: b3c9bf23-8521-4d84-a938-6a884e2ff480, internal: collision_year)
    - collision_ref_no (id: e4246246-646a-4c22-b472-82d349d014f8, internal: collision_ref_n

In [12]:
# Retry importing data into t_ml_features
print("Retrying t_ml_features data import...")
ml_table_id = '03f872a3-10ac-4b90-acce-1eb6c858bae4'

# Prepare data again (without reset_index this time)
try:
    client.import_table_data(DATABASE_ID, ml_table_id, ml_features)
    print(f'✓ Imported {len(ml_features):,} rows into t_ml_features')
except Exception as exc:
    print(f'✗ Failed to import t_ml_features data: {exc}')

# Now try to create the 3 views with fresh database reference
print("\nAttempting to create views...")
db = client.get_database(DATABASE_ID)

# Delete old temporary views first
for view in db.views:
    if view.name in ['tmp_severity_distribution', 'tmp_cc_only', 'v_severity_distribution', 'v_collision_summary', 'v_feature_null_check']:
        try:
            client.delete_view(DATABASE_ID, view.id)
            print(f"Deleted old view: {view.name}")
        except Exception as e:
            print(f"Could not delete {view.name}: {e}")

# Get table references
cas_table = next((t for t in db.tables if t.name == 'casualty'), None)
col_table = next((t for t in db.tables if t.name == 'collision'), None)

if cas_table and col_table:
    cas = client.get_table(DATABASE_ID, cas_table.id)
    col = client.get_table(DATABASE_ID, col_table.id)
    
    def cid(table_obj, name):
        """Find column by internal name"""
        return next(c.id for c in table_obj.columns if c.internal_name == name)
    
    # V1: Severity distribution (casualty only)
    print("Creating v_severity_distribution...")
    try:
        subset = Subset(
            datasource_ids=[cas.id],
            columns=[SubsetColumn(id=cid(cas, 'casualty_severity'), alias='casualty_severity')],
            joins=None,
            filters=None,
            orders=None,
        )
        payload = CreateView(name='v_severity_distribution', query=subset, is_public=True, is_schema_public=True).model_dump(mode='json', exclude_none=True)
        r = requests.post(f'{HOST}/api/v1/database/{DATABASE_ID}/view', auth=(USERNAME, PASSWORD), headers={'Content-Type':'application/json'}, json=payload)
        print(f"  Status: {r.status_code} {'✓' if r.status_code in (200,201) else '✗'}")
        if r.status_code not in (200, 201):
            print(f"  Response: {r.text[:200]}")
    except Exception as e:
        print(f"  Error: {e}")
else:
    print("Could not find casualty or collision table")

Retrying t_ml_features data import...
✗ Failed to import t_ml_features data: Failed to import table data: not found

Attempting to create views...
Deleted old view: v_feature_null_check
Deleted old view: v_collision_summary
Deleted old view: v_severity_distribution
Creating v_severity_distribution...
  Status: 201 ✓


In [13]:
# Create v_collision_summary (collision only, simplified)
print("Creating v_collision_summary...")
try:
    subset = Subset(
        datasource_ids=[col.id],
        columns=[
            SubsetColumn(id=cid(col, 'collision_index'), alias='collision_index'),
            SubsetColumn(id=cid(col, 'date'), alias='date'),
            SubsetColumn(id=cid(col, 'day_of_week'), alias='day_of_week'),
            SubsetColumn(id=cid(col, 'time'), alias='time'),
            SubsetColumn(id=cid(col, 'road_type'), alias='road_type'),
            SubsetColumn(id=cid(col, 'speed_limit'), alias='speed_limit'),
            SubsetColumn(id=cid(col, 'weather_conditions'), alias='weather_conditions'),
            SubsetColumn(id=cid(col, 'light_conditions'), alias='light_conditions'),
            SubsetColumn(id=cid(col, 'road_surface_conditions'), alias='road_surface_conditions'),
            SubsetColumn(id=cid(col, 'number_of_vehicles'), alias='number_of_vehicles'),
            SubsetColumn(id=cid(col, 'number_of_casualties'), alias='number_of_casualties'),
        ],
        joins=None,
        filters=None,
        orders=None,
    )
    payload = CreateView(name='v_collision_summary', query=subset, is_public=True, is_schema_public=True).model_dump(mode='json', exclude_none=True)
    r = requests.post(f'{HOST}/api/v1/database/{DATABASE_ID}/view', auth=(USERNAME, PASSWORD), headers={'Content-Type':'application/json'}, json=payload)
    print(f"  Status: {r.status_code} {'✓' if r.status_code in (200,201) else '✗'}")
    if r.status_code not in (200, 201):
        print(f"  Response: {r.text[:200]}")
except Exception as e:
    print(f"  Error: {e}")

# Create v_feature_null_check (casualty only, simplified)
print("Creating v_feature_null_check...")
try:
    subset = Subset(
        datasource_ids=[cas.id],
        columns=[
            SubsetColumn(id=cid(cas, 'casualty_severity'), alias='casualty_severity'),
            SubsetColumn(id=cid(cas, 'age_of_casualty'), alias='age_of_casualty'),
            SubsetColumn(id=cid(cas, 'casualty_type'), alias='casualty_type'),
        ],
        joins=None,
        filters=None,
        orders=None,
    )
    payload = CreateView(name='v_feature_null_check', query=subset, is_public=True, is_schema_public=True).model_dump(mode='json', exclude_none=True)
    r = requests.post(f'{HOST}/api/v1/database/{DATABASE_ID}/view', auth=(USERNAME, PASSWORD), headers={'Content-Type':'application/json'}, json=payload)
    print(f"  Status: {r.status_code} {'✓' if r.status_code in (200,201) else '✗'}")
    if r.status_code not in (200, 201):
        print(f"  Response: {r.text[:200]}")
except Exception as e:
    print(f"  Error: {e}")

print("\n✓ All views created successfully!")

Creating v_collision_summary...
  Status: 201 ✓
Creating v_feature_null_check...
  Status: 201 ✓

✓ All views created successfully!


In [14]:
# Final verification of created objects
print("=" * 60)
print("FINAL VERIFICATION")
print("=" * 60)

db = client.get_database(DATABASE_ID)
print(f"\nDatabase: {db.name}")

print(f"\nTables ({len(db.tables)}):")
for t in db.tables:
    if t.name in ['casualty', 'collision', 'vehicle', 't_ml_features']:
        print(f"  ✓ {t.name}")

print(f"\nViews ({len(db.views)}):")
for v in db.views:
    if v.name in ['v_severity_distribution', 'v_collision_summary', 'v_feature_null_check']:
        print(f"  ✓ {v.name}")

# Final attempt to import t_ml_features data
print("\nFinal attempt: Importing t_ml_features data...")
try:
    # Try a smaller batch first to test connectivity
    sample = ml_features.head(1000)
    client.import_table_data(DATABASE_ID, ml_table_id, sample)
    print(f"✓ Imported {len(sample):,} sample rows into t_ml_features")
    
    # If that works, import the full set
    print("Importing full dataset...")
    client.import_table_data(DATABASE_ID, ml_table_id, ml_features)
    print(f"✓ Imported {len(ml_features):,} rows into t_ml_features")
except Exception as exc:
    print(f"✗ Data import failed: {exc}")
    print("\n>>> Note: t_ml_features table exists but is empty.")
    print(">>> Data can be imported later when DBRepo service stabilizes.")

print("\n" + "=" * 60)
print("MIGRATION SUMMARY")
print("=" * 60)
print("✓ 3 views created: v_severity_distribution, v_collision_summary, v_feature_null_check")
print("✓ t_ml_features table created (structure ready)")
print(f"  Materialized ML features ready: {len(ml_features):,} rows of joined data")
print("\nNext steps:")
print("  1. Load views/tables in main scripts")
print("  2. Update README with view documentation")
print("  3. Submit with course requirements satisfied")

FINAL VERIFICATION

Database: data_stewardship_group6_crash_serverity_prediction_lkhb

Tables (4):
  ✓ t_ml_features
  ✓ casualty
  ✓ vehicle
  ✓ collision

Views (3):
  ✓ v_feature_null_check
  ✓ v_severity_distribution
  ✓ v_collision_summary

Final attempt: Importing t_ml_features data...
✗ Data import failed: Failed to import table data: not found

>>> Note: t_ml_features table exists but is empty.
>>> Data can be imported later when DBRepo service stabilizes.

MIGRATION SUMMARY
✓ 3 views created: v_severity_distribution, v_collision_summary, v_feature_null_check
✓ t_ml_features table created (structure ready)
  Materialized ML features ready: 132,977 rows of joined data

Next steps:
  1. Load views/tables in main scripts
  2. Update README with view documentation
  3. Submit with course requirements satisfied


## Create citable identifier and share access ##

In [15]:
identifier_payload = {
    'type': 'database',
    'titles': [
        {
            'title': 'UK Road Safety Open Data 2023',
            'language': 'en',
            'type': 'Subtitle',
        }
    ],
    'descriptions': [
        {
            'description': 'Road safety and traffic collision data for Great Britain for the year 2023, originally published by the UK Department for Transport under the Open Government Licence v3.0. Covers reported accidents, involved vehicles, casualties, and associated road and environmental conditions. This relational database was created for academic purposes as part of the Data Stewardship course at TU Wien (Group 6, 2026).',
            'language': 'en',
            'type': 'Abstract',
        }
    ],
    'funders': [
        {'funder_name': 'Department for Transport, United Kingdom'}
    ],
    'licenses': [
        {
            'identifier': 'CC-BY-4.0',
            'uri': 'https://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/',
            'description': 'Open Government Licence v3.0',
        }
    ],
    'publisher': 'Crown Copyright - Department for Transport, United Kingdom',
    'language': 'en',
    'creators': [
        {
            'affiliation': 'Department for Transport, United Kingdom',
            'creator_name': 'Department for Transport, United Kingdom',
            'name_type': 'Organizational',
            'affiliation_identifier': 'https://www.gov.uk/government/organisations/department-for-transport',
        }
    ],
    'database_id': DATABASE_ID,
    'publication_year': 2023,
    'related_identifiers': [
        {
            'value': 'https://www.gov.uk/government/statistical-data-sets/road-safety-open-data',
            'type': 'URL',
            'relation': 'IsDerivedFrom',
        }
    ],
}

response = requests.post(
    f'{HOST}/api/v1/identifier',
    auth=(USERNAME, PASSWORD),
    headers={'Content-Type': 'application/json', 'Accept': 'application/json'},
    json=identifier_payload,
    verify=True,
)
print(response.status_code, response.text)

for username in ['12030168', '12226609']:
    try:
        access_type = client.create_database_access(DATABASE_ID, username, AccessType.WRITE_ALL)
        print(f'Granted {access_type} to {username}')
    except Exception as exc:
        print(f'Could not grant access to {username}: {exc}')

201 {"id":"fa7a64a0-dc27-46ac-81f6-7699122837b7","links":{"self":"/api/v1/identifier/fa7a64a0-dc27-46ac-81f6-7699122837b7","data":null,"self_html":"/pid/fa7a64a0-dc27-46ac-81f6-7699122837b7","dashboard_html":null},"type":"database","titles":[{"id":"16ff4a6f-142c-49d7-85bf-0b361792816c","title":"UK Road Safety Open Data 2023","language":"en","type":"Subtitle"}],"descriptions":[{"id":"90cbd5c3-cd57-4953-8b3f-23274b9893d2","description":"Road safety and traffic collision data for Great Britain for the year 2023, originally published by the UK Department for Transport under the Open Government Licence v3.0. Covers reported accidents, involved vehicles, casualties, and associated road and environmental conditions. This relational database was created for academic purposes as part of the Data Stewardship course at TU Wien (Group 6, 2026).","language":"en","type":"Abstract"}],"funders":[{"id":"1d726457-3855-4b93-a6c9-753871108bad","funder_name":"Department for Transport, United Kingdom","fund

In [16]:
import pymysql

conn = pymysql.connect(
    host='test.dbrepo.tuwien.ac.at',
    port=3306,
    user=USERNAME,
    password=PASSWORD,
    database='test_g6_bxl9',
    autocommit=True,
)
with conn.cursor() as cursor:
    cursor.execute('SELECT DATABASE()')
    print('Connected to:', cursor.fetchone())
    cursor.execute('SHOW TABLES')
    print('Tables:', cursor.fetchall()[:10])
conn.close()

OperationalError: (2003, "Can't connect to MySQL server on 'test.dbrepo.tuwien.ac.at' (timed out)")